## Feature Engineering

## Objective

The goal of this notebook is to transform the cleaned datasets into
ML-ready features and combine useful tourism, hotel, and flight
information into a destination-level dataset.

### Datasets Used

- tourism_data_clean.csv
- hotels_data_clean.csv
- flights_clean.csv
- flight_cost_clean.csv

### Main Output

The final destination-level feature dataset will be saved as:

`data/cleaned/destination_features.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
tourism = pd.read_csv("../data/cleaned/tourism_data_clean.csv")
hotels = pd.read_csv("../data/cleaned/hotels_data_clean.csv")
flights = pd.read_csv("../data/cleaned/flights_clean.csv")
flight_cost = pd.read_csv("../data/cleaned/flight_cost_clean.csv")

print("Tourism:", tourism.shape)
print("Hotels:", hotels.shape)
print("Flights:", flights.shape)
print("Flight Cost:", flight_cost.shape)

In [ ]:
print("Tourism Columns:")
print(tourism.columns.tolist())

print("\nHotels Columns:")
print(hotels.columns.tolist())

print("\nFlights Columns:")
print(flights.columns.tolist())

print("\nFlight Cost Columns:")
print(flight_cost.columns.tolist())

In [ ]:
## Create copies of original datasets

tourism_fe = tourism.copy()
hotels_fe = hotels.copy()
flights_fe = flights.copy()
flight_cost_fe = flight_cost.copy()

### 2. Tourism Feature Engineering

In [ ]:
tourism_fe.info()

In [ ]:
tourism_fe.head()

### 2.1 Convert Family-Friendly Feature

The `Family_Friendly` column contains categorical values such as
Yes/No. These are converted into binary numerical values:

- Yes → 1
- No → 0

In [ ]:
tourism_fe["Family_Friendly"] = (
    tourism_fe["Family_Friendly"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0
    })
)

In [ ]:
tourism_fe["Family_Friendly"].value_counts(dropna=False)

### 2.2 Encode Best Season

`Best_Season` is a categorical feature.

One-hot encoding is used instead of assigning arbitrary numerical
values because seasons do not have a natural numerical order.

In [ ]:
season_dummies = pd.get_dummies(
    tourism_fe["Best_Season"],
    prefix="Season",
    dtype=int
)

tourism_fe = pd.concat(
    [tourism_fe, season_dummies],
    axis=1
)

In [ ]:
tourism_fe.head()

### 2.3 Encode Destination Category

`Category` contains different tourism categories such as Beach,
Adventure, Heritage, Nature, Wildlife, etc.

One-hot encoding is applied so that each category becomes a separate
machine-learning feature.

In [ ]:
category_dummies = pd.get_dummies(
    tourism_fe["Category"],
    prefix="Category",
    dtype=int
)

tourism_fe = pd.concat(
    [tourism_fe, category_dummies],
    axis=1
)

In [ ]:
tourism_fe.head()

## 3. Hotel Feature Engineering

The hotel dataset contains individual hotel-level information.

Instead of keeping every hotel as a separate record, hotel information
will later be aggregated at the city level so that it can be combined
with the destination-level tourism dataset.

In [ ]:
hotels_fe.head()

In [ ]:
hotels_fe.info()

### 3.1 Create Price-to-Rating Feature

A price-to-rating feature is created to represent the approximate
value of a hotel relative to its rating.

The rating is protected against division by zero.

In [ ]:
## Create hotel price per rating

hotels_fe["Price_Per_Rating"] = (
    hotels_fe["Hotel_Price"] /
    hotels_fe["Hotel_Rating"].replace(0, np.nan)
)

### 3.2 Aggregate Hotel Information by City

The tourism dataset is destination-based, while the hotel dataset is
hotel-based.

Therefore, hotel information is aggregated at the city level.

The following features are created:

- Average hotel rating
- Average hotel price
- Minimum hotel price
- Maximum hotel price
- Number of hotels

In [ ]:
## Aggregate hotels by city

hotel_city_features = (
    hotels_fe
    .groupby("City")
    .agg(
        Average_Hotel_Rating=("Hotel_Rating", "mean"),
        Average_Hotel_Price=("Hotel_Price", "mean"),
        Minimum_Hotel_Price=("Hotel_Price", "min"),
        Maximum_Hotel_Price=("Hotel_Price", "max"),
        Number_of_Hotels=("Hotel_Name", "count")
    )
    .reset_index()
)

In [ ]:
hotel_city_features

## 4. Flight Feature Engineering

The flight information is divided across two datasets:

1. `flights_clean.csv`
   - Airline
   - Flight number
   - Origin
   - Destination
   - Schedule information

2. `flight_cost_clean.csv`
   - Origin
   - Destination
   - Average flight cost

The two datasets are combined using origin and destination.

In [ ]:
flights_fe.head()

In [ ]:
flight_cost_fe.head()

### 4.1 Standardize Flight City Names

Origin and destination names are cleaned and standardized so that
they can be consistently used for merging datasets.

In [ ]:

flights_fe["origin"] = (
    flights_fe["origin"]
    .astype(str)
    .str.strip()
    .str.title()
)

flights_fe["destination"] = (
    flights_fe["destination"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [ ]:


flight_cost_fe["origin"] = (
    flight_cost_fe["origin"]
    .astype(str)
    .str.strip()
    .str.title()
)

flight_cost_fe["destination"] = (
    flight_cost_fe["destination"]
    .astype(str)
    .str.strip()
    .str.title()
)

### 4.2 Create Route Feature

A route identifier is created using:

Origin + Destination

Example:

Hyderabad → Goa

becomes:

Hyderabad_TO_Goa

In [ ]:


flights_fe["Route"] = (
    flights_fe["origin"]
    + "_TO_"
    + flights_fe["destination"]
)

In [ ]:
flights_fe[
    ["origin", "destination", "Route"]
].head()

In [ ]:
## Route in flight cost Dataset

flight_cost_fe["Route"] = (
    flight_cost_fe["origin"]
    + "_TO_"
    + flight_cost_fe["destination"]
)

### 4.3 Aggregate Flight Information by Route

Multiple flights can exist between the same origin and destination.

Therefore, route-level features are created:

- Number of flights
- Number of airlines

In [ ]:


flight_route_features = (
    flights_fe
    .groupby(["origin", "destination"])
    .agg(
        Number_of_Flights=("flightNumber", "count"),
        Number_of_Airlines=("airline", "nunique")
    )
    .reset_index()
)

In [ ]:
flight_route_features.head()

### 4.4 Combine Flight Schedule and Flight Cost Data

The route-level flight information is merged with the estimated
average flight cost dataset using:

- origin
- destination

This creates a combined flight feature dataset.

In [ ]:


flight_features = pd.merge(
    flight_route_features,
    flight_cost_fe[
        ["origin", "destination", "avg_flight_cost_inr"]
    ],
    on=["origin", "destination"],
    how="left"
)

In [ ]:
flight_features.head()

## 5. First Attempt: Merge Tourism Destinations with Hotel Cities

The first approach was to directly merge the tourism dataset with
the hotel dataset using:

`Tourism.Destination = Hotels.City`

This approach was tested to determine whether destination names
could directly match hotel city names.

In [ ]:
tourism_fe["Destination"] = (
    tourism_fe["Destination"]
    .astype(str)
    .str.strip()
    .str.title()
)

hotel_city_features["City"] = (
    hotel_city_features["City"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [ ]:
destination_data = pd.merge(
    tourism_fe,
    hotel_city_features,
    left_on="Destination",
    right_on="City",
    how="left"
)

In [ ]:
destination_data.head()

In [ ]:
print(
    "Destinations:",
    destination_data["Destination"].nunique()
)

print(
    "Destinations with hotel data:",
    destination_data["Average_Hotel_Price"].notna().sum()
)

print(
    "Destinations without hotel data:",
    destination_data["Average_Hotel_Price"].isna().sum()
)

In [ ]:
destination_data[
    destination_data["Average_Hotel_Price"].isna()
]["Destination"].unique()

## 6. Problem Identified: Destination and Hotel City Mismatch

The direct merge produced very poor coverage.

There are 92 tourism destinations, but only a small number matched
the 51 hotel cities.

The reason is that the tourism dataset contains tourist attractions
and regions, while the hotel dataset is organized mainly by cities.

Examples:

- `Ladakh (Leh)` vs `Leh`
- `Kerala Backwaters (Alappuzha & Kumarakom)` vs `Kumarakom`
- `Coorg (Kodagu)` vs a hotel city
- `Varanasi (Kashi)` vs `Varanasi`

Therefore, directly matching `Destination` with `City` is not a
reliable approach.

The first merge is rejected and a controlled mapping approach is used.

In [ ]:
hotel_cities = sorted(
    hotels_fe["City"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.title()
    .unique()
)

print("Number of hotel cities:", len(hotel_cities))
print("Hotel Cities: ",hotel_cities)

In [ ]:
tourism_locations = (
    tourism_fe[["Destination", "State"]]
    .drop_duplicates()
    .sort_values("Destination")
)

display(tourism_locations)

In [ ]:
destination_to_hotel_city = {
    "Varanasi (Kashi)": "Varanasi",
    "Varanasi Ganga Ghats": "Varanasi",
    "Kerala Backwaters (Alappuzha & Kumarakom)": "Kumarakom",
    "Kashmir": "Kashmir",
    "Sikkim": "Sikkim",
    "Trivandrum": "Trivandrum",
    "Pondicherry": "Pondicherry"
}

In [ ]:
hotel_city_set = set(hotel_cities)

tourism_fe["Hotel_City"] = tourism_fe["Destination"].where(
    tourism_fe["Destination"].isin(hotel_city_set)
)

In [ ]:
tourism_fe["Hotel_City"] = tourism_fe["Destination"].map(
    destination_to_hotel_city
).fillna(tourism_fe["Hotel_City"])

In [ ]:
print(tourism_fe.columns.tolist())

print(
    tourism_fe[
        ["Destination", "State", "Hotel_City"]
    ].head(20)
)

In [ ]:
mapping_result = tourism_fe[
    ["Destination", "State", "Hotel_City"]
].sort_values("Destination")

display(mapping_result)

In [ ]:
print(
    "Total destinations:",
    len(tourism_fe)
)

print(
    "Destinations mapped to hotel city:",
    tourism_fe["Hotel_City"].notna().sum()
)

print(
    "Destinations without hotel city:",
    tourism_fe["Hotel_City"].isna().sum()
)

In [ ]:
unmapped_destinations = tourism_fe[
    tourism_fe["Hotel_City"].isna()
][["Destination", "State"]]

print(unmapped_destinations)

In [ ]:
destination_data = pd.merge(
    tourism_fe,
    hotel_city_features,
    left_on="Hotel_City",
    right_on="City",
    how="left"
)

In [ ]:
destination_data.drop(
    columns=["City"],
    inplace=True
)

In [ ]:
destination_data[
    [
        "Destination",
        "State",
        "Hotel_City",
        "Average_Hotel_Rating",
        "Average_Hotel_Price"
    ]
].head(20)

In [ ]:
print("Tourism airports:")
print(
    tourism_fe["Nearest_Airport"]
    .dropna()
    .unique()
)

In [ ]:
print("Tourism airports:")
print(
    tourism_fe["Nearest_Airport"]
    .dropna()
    .unique()
)

In [ ]:
print("Flight destinations:")
print(
    flights_fe["destination"]
    .dropna()
    .unique()
)

In [ ]:
print(
    flight_cost_fe[
        ["origin", "destination", "avg_flight_cost_inr"]
    ].drop_duplicates()
)

In [ ]:
tourism_fe["Airport_City"] = (
    tourism_fe["Nearest_Airport"]
    .astype(str)
    .str.replace(" Airport", "", regex=False)
    .str.strip()
)

In [ ]:
tourism_fe[
    ["Destination", "Nearest_Airport", "Airport_City"]
].head(20)

In [ ]:
city_mapping = {
    "Bangalore": "Bengaluru",
    "Bengaluru": "Bengaluru",
    "Bombay": "Mumbai",
    "Madras": "Chennai",
    "Calcutta": "Kolkata"
}

In [ ]:
tourism_fe["Airport_City"] = (
    tourism_fe["Airport_City"]
    .replace(city_mapping)
)

flights_fe["origin"] = (
    flights_fe["origin"]
    .astype(str)
    .str.strip()
    .replace(city_mapping)
)

flights_fe["destination"] = (
    flights_fe["destination"]
    .astype(str)
    .str.strip()
    .replace(city_mapping)
)

flight_cost_fe["origin"] = (
    flight_cost_fe["origin"]
    .astype(str)
    .str.strip()
    .replace(city_mapping)
)

flight_cost_fe["destination"] = (
    flight_cost_fe["destination"]
    .astype(str)
    .str.strip()
    .replace(city_mapping)
)

In [ ]:
flight_destination_set = set(
    flights_fe["destination"].dropna().unique()
)

tourism_fe["Flight_Available"] = (
    tourism_fe["Airport_City"]
    .isin(flight_destination_set)
    .astype(int)
)

In [ ]:
print(
    "Destinations with flight coverage:",
    tourism_fe["Flight_Available"].sum()
)

print(
    "Destinations without flight coverage:",
    (tourism_fe["Flight_Available"] == 0).sum()
)

In [ ]:
tourism_fe[
    tourism_fe["Flight_Available"] == 0
][
    ["Destination", "Nearest_Airport", "Airport_City"]
]

In [ ]:
flight_route_features = (
    flights_fe
    .groupby(["origin", "destination"])
    .agg(
        Number_of_Flights=("flightNumber", "count"),
        Number_of_Airlines=("airline", "nunique")
    )
    .reset_index()
)

In [ ]:
flight_features = pd.merge(
    flight_route_features,
    flight_cost_fe[
        ["origin", "destination", "avg_flight_cost_inr"]
    ],
    on=["origin", "destination"],
    how="left"
)

In [ ]:
flight_features.head(20)

In [ ]:
destination_flight_features = (
    flight_cost_fe
    .groupby("destination")
    .agg(
        Average_Flight_Cost=("avg_flight_cost_inr", "mean"),
        Minimum_Flight_Cost=("avg_flight_cost_inr", "min"),
        Maximum_Flight_Cost=("avg_flight_cost_inr", "max"),
        Number_of_Routes=("origin", "nunique")
    )
    .reset_index()
)

In [ ]:
destination_flight_features.head(20)

In [ ]:
destination_data = pd.merge(
    tourism_fe,
    hotel_city_features,
    left_on="Hotel_City",
    right_on="City",
    how="left"
)

destination_data.drop(
    columns=["City"],
    inplace=True
)

In [ ]:
print(destination_data.columns.tolist())

In [ ]:
destination_data = pd.merge(
    destination_data,
    destination_flight_features,
    left_on="Airport_City",
    right_on="destination",
    how="left"
)

In [ ]:
destination_data.drop(
    columns=["destination"],
    inplace=True
)

In [ ]:
destination_data[
    [
        "Destination",
        "Airport_City",
        "Flight_Available",
        "Average_Flight_Cost",
        "Minimum_Flight_Cost",
        "Maximum_Flight_Cost",
        "Number_of_Routes"
    ]
].head(20)

In [ ]:
print("Total destinations:", destination_data["Destination"].nunique())

print(
    "Destinations with flight data:",
    destination_data["Average_Flight_Cost"].notna().sum()
)

print(
    "Destinations without flight data:",
    destination_data["Average_Flight_Cost"].isna().sum()
)

In [ ]:
print(
    destination_data["Flight_Available"].value_counts()
)

In [ ]:
destination_data["Hotel_Data_Available"] = (
    destination_data["Average_Hotel_Price"].notna().astype(int)
)

destination_data["Supporting_Data_Available"] = (
    (
        destination_data["Hotel_Data_Available"] == 1
    )
    |
    (
        destination_data["Flight_Available"] == 1
    )
).astype(int)

In [ ]:
destination_data[
    [
        "Destination",
        "Hotel_Data_Available",
        "Flight_Available",
        "Supporting_Data_Available"
    ]
].head(20)

In [ ]:
print("Shape:", destination_data.shape)

print("\nColumns:")
print(destination_data.columns.tolist())

print("\nMissing values:")
print(destination_data.isnull().sum())

In [ ]:
destination_data.to_csv(
    "../data/cleaned/destination_features.csv",
    index=False
)

print("Day 3 dataset saved successfully.")

In [ ]:
destination_data.columns